# Bootstrap triple-collocation analysis

Run the final 20 controlled CONUS triplets. ASCAT-target experiments fix SMAP-EF as the opposite-sensor member; SMAP-target experiments fix ASCAT-EF. ERA5-Land and the NLDAS Noah, VIC, and Mosaic products are evaluated as separate land-reference members.

There is one fixed full comparison period (2016-03-31 through 2023-12-31), not separate Full/Test options. Only the 5,000-member IID bootstrap calculation is retained. Each output contains final bootstrap-mean error standard deviation, final bootstrap-mean TCA correlation, valid-bootstrap counts, explicit triplet-member metadata, and the shared bootstrap-index SHA-256 hash; Raw/Masked/Merged/single-run products are not saved.

In [ ]:
from config import configure_runtime

configure_runtime()

In [ ]:
import os
import tempfile
from pathlib import Path

from config import (
    TCA_WORKERS,
    base_FP,
    cpuserver_data,
    nas_FP,
    das_FP,
    george_FP,
)
from TCA import (
    FIXED_TRIPLETS,
    ProductSpec,
    run_bootstrap_inventory,
    summarize_fixed_inventory,
)

from config import figures_FP, results_FP


## Scientific settings

The 101-day centered rolling mean is removed before ETC. Each IID bootstrap draws half of the aligned dates with replacement. A solution requires at least 25 joint observations, nonnegative error variances, fractional MSE within [0, 1], and nonnegative pairwise correlations. Final grids require at least 2,500 valid solutions (50% of 5,000).

In [ ]:
COMPARISON_START = "2016-03-31"
COMPARISON_END = "2023-12-31"
CONUS_BOUNDS = (-126.0, -66.0, 24.0, 51.0)
ROLLING_WINDOW_DAYS = 101
BOOTSTRAP_COUNT = 5000
BOOTSTRAP_FRACTION = 0.5
BOOTSTRAP_RANDOM_SEED = 42
MINIMUM_JOINT_OBSERVATIONS = 25
MINIMUM_PAIRWISE_CORRELATION = 0.0
MINIMUM_VALID_BOOTSTRAP_FRACTION = 0.5

# Memory/performance controls do not change the scientific calculation.
BOOTSTRAP_BATCH_SIZE = 10
CHUNK_LATITUDE = 10
CHUNK_LONGITUDE = 60
ANOMALY_CHUNK_LATITUDE = 10
ANOMALY_CHUNK_LONGITUDE = 60

## Inputs and compact output location

In [ ]:
RESULT_FP = Path(results_FP)
PREDICTION_FP = RESULT_FP / "CONUS_Prediction"
TCA_RESULT_FP = RESULT_FP / "TCA"
TCA_SUMMARY_FILE = TCA_RESULT_FP / "TCA_summary.nc"
TCA_SCRATCH_FP = Path(
    os.environ.get("RZSM_TCA_SCRATCH", tempfile.gettempdir())
)

PRODUCT_SPECS = {
    "ASCAT_EF": ProductSpec(PREDICTION_FP / "Fixed_EF_ASCAT_prediction.nc", "RZSM_prediction", "RZSM_EF_ASCAT"),
    "ASCAT_FNO": ProductSpec(PREDICTION_FP / "FNO_ASCAT_prediction.nc", "RZSM_prediction", "RZSM_FNO_ASCAT"),
    "ASCAT_LSTM": ProductSpec(PREDICTION_FP / "LSTM_ASCAT_prediction.nc", "RZSM_prediction", "RZSM_LSTM_ASCAT"),
    "SMAP_EF": ProductSpec(PREDICTION_FP / "Fixed_EF_SMAP_prediction.nc", "RZSM_prediction", "RZSM_EF_SMAP"),
    "SMAP_FNO": ProductSpec(PREDICTION_FP / "FNO_SMAP_prediction.nc", "RZSM_prediction", "RZSM_FNO_SMAP"),
    "SMAP_LSTM": ProductSpec(PREDICTION_FP / "LSTM_SMAP_prediction.nc", "RZSM_prediction", "RZSM_LSTM_SMAP"),
    "ERA5-Land": ProductSpec(RESULT_FP / "ERA5-Land" / "ERA5_Land_20150401_20231231_eqd_010.nc", "RZSM", "RZSM_ERA5Land"),
    "NLDAS_NOAH": ProductSpec(RESULT_FP / "NLDAS" / "NLDAS_NOAH_20150401_20231231_eqd_010.nc", "RZSM", "RZSM_NLDAS_NOAH"),
    "NLDAS_VIC": ProductSpec(RESULT_FP / "NLDAS" / "NLDAS_VIC_20150401_20231231_eqd_010.nc", "RZSM", "RZSM_NLDAS_VIC"),
    "NLDAS_MOSAIC": ProductSpec(RESULT_FP / "NLDAS" / "NLDAS_MOSAIC_20150401_20231231_eqd_010.nc", "RZSM", "RZSM_NLDAS_MOSAIC"),
}
TCA_RESULT_FP.mkdir(parents=True, exist_ok=True)

In [ ]:
required_keys = {
    key
    for triplet in FIXED_TRIPLETS
    for key in (triplet.ascat, triplet.smap, triplet.reference)
}
missing_files = [
    Path(PRODUCT_SPECS[key].path)
    for key in sorted(required_keys)
    if not Path(PRODUCT_SPECS[key].path).is_file()
]
if missing_files:
    raise FileNotFoundError("Missing TCA inputs:\n" + "\n".join(map(str, missing_files)))
print(f"Triplets: {len(FIXED_TRIPLETS)}")
print(f"Workers: {TCA_WORKERS}")
print(f"Final output directory: {TCA_RESULT_FP}")

## Run final bootstrap TCA

Rolling-anomaly arrays and bootstrap indices are temporary scratch data and are removed automatically after all ten triplets finish.

In [ ]:
tca_files = run_bootstrap_inventory(
    triplets=FIXED_TRIPLETS,
    product_specs=PRODUCT_SPECS,
    output_directory=TCA_RESULT_FP,
    scratch_directory=TCA_SCRATCH_FP,
    start_date=COMPARISON_START,
    end_date=COMPARISON_END,
    bounds=CONUS_BOUNDS,
    bootstrap_count=BOOTSTRAP_COUNT,
    bootstrap_fraction=BOOTSTRAP_FRACTION,
    random_seed=BOOTSTRAP_RANDOM_SEED,
    rolling_window=ROLLING_WINDOW_DAYS,
    nod_threshold=MINIMUM_JOINT_OBSERVATIONS,
    correlation_threshold=MINIMUM_PAIRWISE_CORRELATION,
    min_valid_bootstrap_fraction=MINIMUM_VALID_BOOTSTRAP_FRACTION,
    workers=TCA_WORKERS,
    bootstrap_batch_size=BOOTSTRAP_BATCH_SIZE,
    chunk_latitude=CHUNK_LATITUDE,
    chunk_longitude=CHUNK_LONGITUDE,
    anomaly_chunk_latitude=ANOMALY_CHUNK_LATITUDE,
    anomaly_chunk_longitude=ANOMALY_CHUNK_LONGITUDE,
)
summarize_fixed_inventory(
    tca_paths=tca_files,
    triplets=FIXED_TRIPLETS,
    product_specs=PRODUCT_SPECS,
    output_file=TCA_SUMMARY_FILE,
)